# LangChain Fundamentals - Groq Edition

**Stack:** Groq API + `llama-3.3-70b-versatile`  
**Original source:** [gkamradt/langchain-tutorials](https://github.com/gkamradt/langchain-tutorials)  
**Author:** Zariya Shahid  

This notebook covers the core building blocks of LangChain, rewritten from scratch using Groq instead of OpenAI.  
Topics covered:
- Schema (Text, Chat Messages, Documents)
- Models (LLM, Chat Model)
- Prompts (PromptTemplate, Output Parsers)
- Indexes (Document Loaders, Text Splitters, Retrievers, VectorStores)
- Memory
- Chains
- Agents

## Setup

In [ ]:
# Install dependencies (run once)
!pip install langchain langchain-groq langchain-community langchain-text-splitters python-dotenv chromadb sentence-transformers faiss-cpu

In [ ]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Key loaded:", GROQ_API_KEY[:8], "...")

---
## 1. Schema - The Nuts and Bolts

Schema defines the basic data types LangChain works with.

### 1.1 Text
The simplest form - just a plain string passed to the model.

In [ ]:
my_text = "What is the capital of Pakistan?"
print(my_text)

### 1.2 Chat Messages
LangChain uses three message types:
- `SystemMessage` - sets behavior/context for the model
- `HumanMessage` - the user's input
- `AIMessage` - the model's previous response (used for multi-turn conversation)

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

chat = ChatGroq(
    temperature=0.7,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

In [ ]:
# Basic system + human message
response = chat.invoke(
    [
        SystemMessage(content="You are a helpful assistant that answers in one short sentence."),
        HumanMessage(content="What programming language should I learn first?")
    ]
)
print(response.content)

In [ ]:
# Multi-turn: pass prior AI response as AIMessage to continue the conversation
response = chat.invoke(
    [
        SystemMessage(content="You are a helpful coding mentor."),
        HumanMessage(content="What should I learn after Python?"),
        AIMessage(content="After Python, I recommend learning JavaScript for web development."),
        HumanMessage(content="What JavaScript framework is best for beginners?")
    ]
)
print(response.content)

### 1.3 Documents
A `Document` object holds a chunk of text plus optional metadata (source, date, ID, etc.).

In [ ]:
from langchain_core.documents import Document

# Document with metadata
doc = Document(
    page_content="LangChain is a framework for building LLM-powered applications.",
    metadata={
        "source": "LangChain Docs",
        "created_at": "2024-01-01",
        "doc_id": 101
    }
)
print(doc)
print("\nContent:", doc.page_content)
print("Metadata:", doc.metadata)

In [ ]:
# Document without metadata is also valid
doc_simple = Document(page_content="Pakistan was founded in 1947.")
print(doc_simple)

---
## 2. Models - Talking to the AI Brain

LangChain supports two model types:
- **LLM** - text in, text out (completion style)
- **Chat Model** - messages in, message out (conversation style)

### 2.1 Chat Model (Groq)
We use `ChatGroq` throughout this notebook. Groq runs `llama-3.3-70b-versatile` at extremely fast inference speeds.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# temperature=0 for deterministic output, 1 for creative
chat = ChatGroq(
    temperature=1,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

response = chat.invoke(
    [
        SystemMessage(content="You are a sarcastic assistant that always complains."),
        HumanMessage(content="Can you help me sort a list in Python?")
    ]
)
print(response.content)

---
## 3. Prompts - Instructing the Model

Prompts define how you communicate with the model. LangChain provides templates to make this dynamic and reusable.

### 3.1 PromptTemplate
Think of it as an f-string for LLM prompts - define a template with placeholders, fill them at runtime.

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are a senior software engineer.
Explain what {concept} is in simple terms, in 2-3 sentences.
"""

prompt = PromptTemplate(
    input_variables=["concept"],
    template=template
)

# Format fills in the placeholder
final_prompt = prompt.format(concept="RAG (Retrieval Augmented Generation)")
print("Final Prompt:\n", final_prompt)

In [ ]:
# Now send the formatted prompt to Groq
chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

response = chat.invoke([HumanMessage(content=final_prompt)])
print(response.content)

### 3.2 ChatPromptTemplate
For chat models, use `ChatPromptTemplate` to build multi-role prompt templates.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert in {domain}. Answer concisely."),
        ("human", "{question}")
    ]
)

messages = chat_template.format_messages(
    domain="AI and machine learning",
    question="What is the difference between fine-tuning and RAG?"
)

response = chat.invoke(messages)
print(response.content)

### 3.3 Output Parsers
Output parsers let you extract structured data from LLM responses instead of getting raw text.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

# Define the structure you want back
class LanguageInfo(BaseModel):
    language: str = Field(description="The programming language name")
    use_case: str = Field(description="What it is best used for")
    difficulty: str = Field(description="Beginner, Intermediate, or Advanced")

output_parser = JsonOutputParser(pydantic_object=LanguageInfo)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

In [ ]:
# Build prompt that includes format instructions
prompt = PromptTemplate(
    template="Tell me about this programming language.\n{format_instructions}\nLanguage: {language}",
    input_variables=["language"],
    partial_variables={"format_instructions": format_instructions}
)

chain = prompt | chat | output_parser

result = chain.invoke({"language": "Python"})
print(result)
print("\nType:", type(result))
print("Language:", result["language"])
print("Use case:", result["use_case"])
print("Difficulty:", result["difficulty"])

---
## 4. Indexes - Working with Documents

Indexes let you load, split, embed, and retrieve documents. This is the foundation of RAG.

### 4.1 Text Splitters
Large documents must be split into smaller chunks before embedding.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """
LangChain is a framework designed to simplify the creation of applications using large language models.
It provides tools for chaining together components like prompts, models, memory, and tools.
One of its key strengths is the ability to connect LLMs with external data sources.
You can load documents from PDFs, websites, databases, and more.
LangChain supports multiple LLM providers including OpenAI, Anthropic, Groq, and HuggingFace.
The framework also supports agents, which can use tools to take actions based on LLM reasoning.
Memory modules let you persist conversation history across multiple turns.
RAG (Retrieval Augmented Generation) is one of the most popular patterns built with LangChain.
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

chunks = text_splitter.create_documents([long_text])
print(f"Split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:100]}...")

### 4.2 Embeddings and VectorStore
Embeddings convert text into vectors so we can search by semantic similarity.

In [ ]:
# We use sentence-transformers for free local embeddings (no API key needed)
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Sample knowledge base
documents = [
    "Python is a high-level, general-purpose programming language known for its simplicity.",
    "JavaScript is mainly used for web development and runs in the browser.",
    "Rust is a systems programming language focused on memory safety and performance.",
    "SQL is used for managing and querying relational databases.",
    "TypeScript is a superset of JavaScript that adds static typing."
]

# Create embeddings using a free local model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in FAISS vector store
vectorstore = FAISS.from_texts(documents, embeddings)

print("VectorStore created successfully.")

In [ ]:
# Retrieve the most relevant documents for a query
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

query = "Which language is best for web development?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}")

---
## 5. Memory - Giving the Model a Short-Term Memory

By default, LLMs have no memory between calls. LangChain's memory modules let you persist conversation history.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

# Simulate a conversation
history.add_ai_message("Hello! I am your coding assistant. What are you working on?")
history.add_user_message("I am building a RAG pipeline.")

print("Current history:")
for msg in history.messages:
    print(f"  [{msg.type}]: {msg.content}")

In [ ]:
# Send history to the model and get a response
chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

ai_response = chat.invoke(history.messages)
print("AI Response:", ai_response.content)

# Add the response back to history for future turns
history.add_ai_message(ai_response.content)

print("\nUpdated history length:", len(history.messages), "messages")

---
## 6. Chains - Connecting LLM Calls Together

Chains let you pass the output of one LLM call as the input to another, automating multi-step reasoning.

### 6.1 Simple Sequential Chain (LCEL style)
Chain 1 output feeds directly into Chain 2 input using the modern pipe operator.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

chat = ChatGroq(
    temperature=0.7,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Chain 1: Given a topic, suggest a project idea
template1 = """
Given a tech topic, suggest one beginner-friendly project idea in one sentence.

Topic: {topic}
Project idea:
"""
prompt1 = PromptTemplate(input_variables=["topic"], template=template1)

# Chain 2: Given a project idea, list the tech stack
template2 = """
Given this project idea, list the ideal tech stack (3-4 technologies only).

Project: {project}
Tech stack:
"""
prompt2 = PromptTemplate(input_variables=["project"], template=template2)

# LCEL sequential chain using pipe operator
chain1 = prompt1 | chat | StrOutputParser()
chain2 = prompt2 | chat | StrOutputParser()

# Run chain 1 then pass its output into chain 2
project_idea = chain1.invoke({"topic": "LangChain"})
print("Project idea:", project_idea)

tech_stack = chain2.invoke({"project": project_idea})
print("\nTech stack:", tech_stack)

### 6.2 LCEL (LangChain Expression Language) - Modern Chain Style
LCEL uses the `|` pipe operator to chain components. This is the preferred modern approach.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

prompt = ChatPromptTemplate.from_template(
    "Explain {concept} in 2 sentences like I'm 5 years old."
)

# LCEL chain: prompt | model | output parser
chain = prompt | chat | StrOutputParser()

result = chain.invoke({"concept": "embeddings"})
print(result)

In [ ]:
# You can reuse the same chain with different inputs
for concept in ["vector databases", "LLM temperature", "tokenization"]:
    result = chain.invoke({"concept": concept})
    print(f"[{concept}]")
    print(result)
    print()

---
## 7. Agents - LLMs That Make Decisions

Agents use the LLM as a reasoning engine to decide which tool to call and when. Instead of a fixed chain, the model decides the next step dynamically.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

# Define tools
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

@tool
def reverse_string(text: str) -> str:
    """Reverses a given string."""
    return text[::-1]

tools = [get_word_length, reverse_string]

# Create the chat model
chat = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY
)

# Create the agent using LangGraph (recommended in LangChain 1.x)
agent = create_react_agent(chat, tools)

# The agent will decide which tool to use
result = agent.invoke({"messages": [{"role": "user", "content": "What is the length of the word 'LangChain' and what does it look like reversed?"}]})
print("\nFinal Answer:", result["messages"][-1].content)

---
## Summary

| Component | What it does |
|---|---|
| Schema | Basic data types: Text, Chat Messages, Documents |
| Models | Interface to the LLM (ChatGroq in our case) |
| Prompts | Templates and structured instructions for the model |
| Output Parsers | Extract structured data from LLM responses |
| Text Splitters | Split large documents into LLM-friendly chunks |
| VectorStores | Store and search document embeddings |
| Retrievers | Fetch relevant documents for a query |
| Memory | Persist conversation history across turns |
| Chains | Connect multiple LLM calls together (LCEL pipe style) |
| Agents | LLM decides which tools to call dynamically |

**Next up:** Step 2 - Memory and Conversation Chains in depth.